# Agentic GraphRAG demo: Agent + GraphRAG with Gemini and Neo4j

**GraphRAG (Day 11)** ran a fixed pipeline: question → one Cypher query → answer.
**Agentic GraphRAG** puts an **agent** in charge of retrieval. For each question it decides:
- search the **document chunks** (vector search + connected graph facts), or
- query the **knowledge graph** with Cypher, or
- do both, one step at a time, until it can answer.

**Two phases**
| Phase | Steps |
|---|---|
| **Indexing** (once) | Documents → chunks → Gemini extracts entities & relationships → Neo4j graph + chunk embeddings |
| **Query** (every question) | Question → agent → tools (vector search / Cypher) → observe → answer with sources |

**Before you start**
- Neo4j running locally with the **APOC** plugin
- A `.env` file next to this notebook:
```
GOOGLE_API_KEY=your-google-ai-studio-key
NEO4J_URI=neo4j://127.0.0.1:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=your-neo4j-password
GEMINI_MODEL=gemini-3.6-flash
GEMINI_EMBED_MODEL=models/gemini-embedding-001
```
⚠️ Step 3 **clears the database**, so the demo can be re-run.

## Step 0 · Install packages

In [ ]:
%pip install -q "langchain>=1.0" langchain-community langchain-text-splitters langchain-neo4j langchain-experimental langchain-google-genai python-dotenv

## Step 1 · Settings from `.env`

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)   # always take the latest values from .env

NEO4J_URI = os.getenv("NEO4J_URI", "neo4j://127.0.0.1:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD")
GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
GEMINI_EMBED_MODEL = os.getenv("GEMINI_EMBED_MODEL", "models/gemini-embedding-001")

assert os.getenv("GOOGLE_API_KEY"), "GOOGLE_API_KEY is missing from .env"
assert NEO4J_PASSWORD, "NEO4J_PASSWORD is missing from .env"
print("Settings loaded:", NEO4J_URI, GEMINI_MODEL, GEMINI_EMBED_MODEL)

## Step 2 · Connect to Neo4j and Gemini
- `llm` is used to **extract** the graph and to run the **agent**
- `embeddings` turns chunks and questions into vectors for **semantic search**
- The rate limiter keeps us under the free-tier requests-per-minute limit

In [ ]:
from langchain_neo4j import Neo4jGraph
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.rate_limiters import InMemoryRateLimiter

graph = Neo4jGraph(url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD)

llm = ChatGoogleGenerativeAI(
    model=GEMINI_MODEL,
    temperature=0,
    max_retries=2,
    rate_limiter=InMemoryRateLimiter(requests_per_second=0.1, max_bucket_size=1),  # ~6 calls/minute
)
embeddings = GoogleGenerativeAIEmbeddings(model=GEMINI_EMBED_MODEL)
print("Connected")

# Phase 1 · Indexing: documents → knowledge graph + vectors

## Step 3 · Two small input documents

In [ ]:
graph.query("MATCH (n) DETACH DELETE n")   # start clean
os.makedirs("data", exist_ok=True)

doc1 = """Alice is a data engineer at Neo4j since 2021. Alice lives in London.
Ravi is a data scientist at Neo4j since 2023. Ravi lives in Bengaluru.
Bob is a machine learning engineer at Google since 2019. Bob lives in Bengaluru.
Alice and Bob both studied at IIT Hyderabad. Ravi studied at IIT Madras.
Neo4j has offices in London and Bengaluru. Google has offices in Bengaluru and Hyderabad."""

doc2 = """The GraphRAG project helps support teams answer customer questions by combining
a knowledge graph with document search, so answers come with the facts that support them.
Alice and Ravi work on the GraphRAG project. The GraphRAG project uses Gemini.
The Gemini project builds large language models that understand text, images and code.
Bob works on the Gemini project."""

open("data/team.txt", "w").write(doc1)
open("data/projects.txt", "w").write(doc2)
print("Created data/team.txt and data/projects.txt")

## Step 4 · Load and chunk

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

docs = TextLoader("data/team.txt").load() + TextLoader("data/projects.txt").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40).split_documents(docs)
print(f"{len(docs)} documents -> {len(chunks)} chunks")

## Step 5 · Gemini extracts entities, relationships and properties
`LLMGraphTransformer` sends each chunk to Gemini and gets back **GraphDocuments** (nodes + relationships + source chunk).

In [ ]:
from langchain_experimental.graph_transformers import LLMGraphTransformer

transformer = LLMGraphTransformer(
    llm=llm,
    allowed_nodes=["Person", "Company", "City", "University", "Project"],
    allowed_relationships=["WORKS_AT", "LIVES_IN", "STUDIED_AT",
                           "HAS_OFFICE_IN", "WORKS_ON", "USES"],
    node_properties=["role"],
    relationship_properties=["since"],
)
graph_docs = transformer.convert_to_graph_documents(chunks)

for gd in graph_docs:
    for n in gd.nodes:
        print(f"NODE {n.id} ({n.type}) {n.properties}")
    for r in gd.relationships:
        print(f"REL  {r.source.id} -[{r.type}]-> {r.target.id} {r.properties}")

## Step 6 · Store the graph in Neo4j
`add_graph_documents` writes everything with Cypher `MERGE`.
`include_source=True` stores each chunk as a `Document` node linked to its entities with `MENTIONS`.

In [ ]:
graph.add_graph_documents(graph_docs, include_source=True, baseEntityLabel=True)
graph.refresh_schema()
print(graph.schema)

## Step 7 · Add a vector index on the chunks
Each `Document` (chunk) node gets an `embedding` property, and Neo4j builds a vector index on it.
The graph and the vectors now live in **one database**.

The `retrieval_query` is the "Graph" part of the search: for every similar chunk it also
follows `MENTIONS` to the entities in that chunk and returns their relationships.

In [ ]:
from langchain_neo4j import Neo4jVector

Neo4jVector.from_existing_graph(
    embedding=embeddings,
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD,
    index_name="chunk_index",
    node_label="Document",
    text_node_properties=["text"],
    embedding_node_property="embedding",
)

retrieval_query = """
OPTIONAL MATCH (node)-[:MENTIONS]->(e)-[r]->(other)
WHERE NOT other:Document
WITH node, score, collect(DISTINCT e.id + ' -' + type(r) + '-> ' + other.id) AS facts
RETURN node.text + '\\nGraph facts: ' + reduce(s = '', f IN facts | s + f + '; ') AS text,
       score, {source: node.source} AS metadata
"""

chunk_store = Neo4jVector.from_existing_index(
    embedding=embeddings,
    url=NEO4J_URI, username=NEO4J_USERNAME, password=NEO4J_PASSWORD,
    index_name="chunk_index",
    retrieval_query=retrieval_query,
)

# Quick test of the vector + graph search (no LLM call)
for d in chunk_store.similarity_search("What does the GraphRAG project do?", k=1):
    print(d.page_content)

# Phase 2 · Query: the agent decides how to retrieve

## Step 8 · Give the agent three GraphRAG tools
| Tool | Retrieval type | Good for |
|---|---|---|
| `search_documents` | Vector search + graph facts from `MENTIONS` | "What is / why / describe" questions |
| `find_entity` | Graph lookup (case-insensitive) | Getting the exact stored name |
| `query_graph` | Read-only Cypher written by the agent | Exact facts, multi-hop, counting |

The **docstrings** are how the agent decides which tool to use.

In [ ]:
import re
from langchain_core.tools import tool

@tool
def search_documents(query: str) -> str:
    """Semantic search over the document chunks. Returns the most relevant text
    plus graph facts about the entities it mentions. Use for descriptive questions
    (what is, why, how, describe) or when you need the original wording."""
    results = chunk_store.similarity_search(query, k=2)
    return "\n\n".join(f"[source: {d.metadata.get('source')}]\n{d.page_content}" for d in results)

@tool
def find_entity(name: str) -> list:
    """Find entities whose name contains the text (case-insensitive).
    Use this to get the exact stored name before writing Cypher."""
    return graph.query("""
        MATCH (n:__Entity__) WHERE toLower(n.id) CONTAINS toLower($name)
        RETURN n.id AS name, labels(n) AS labels LIMIT 10
    """, params={"name": name})

WRITE_WORDS = re.compile(r"\b(CREATE|MERGE|DELETE|DETACH|SET|REMOVE|DROP|LOAD|CALL)\b", re.IGNORECASE)

@tool
def query_graph(cypher: str) -> list | str:
    """Run a READ-ONLY Cypher query (MATCH ... RETURN ...) on the knowledge graph.
    Use for exact facts, relationships between entities, multi-hop questions and counts.
    Entity names are in the `id` property; use exact names from find_entity."""
    if WRITE_WORDS.search(cypher):
        return "Rejected: only read-only MATCH/RETURN queries are allowed."
    try:
        return graph.query(cypher)[:25]
    except Exception as e:
        return f"Cypher error: {e}. Fix the query and try again."

tools = [search_documents, find_entity, query_graph]

## Step 9 · Create the agent
The graph **schema** goes into the system prompt, so the agent can write correct Cypher
without spending a tool call to look it up.

In [ ]:
from langchain.agents import create_agent

SYSTEM_PROMPT = f"""You answer questions using a GraphRAG knowledge base in Neo4j.

Graph schema:
{graph.schema}

Rules:
- Entity names are stored in the `id` property. Use find_entity to get exact names.
- For exact facts, relationships and multi-hop questions, use query_graph.
- For descriptions or explanations, use search_documents.
- For mixed questions, use both, one step at a time.
- Answer only from tool results. If the information is not there, say so.
- End with 'Sources:' listing the graph paths and/or source files you used."""

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

A helper that prints every step, so students can see **which retrieval the agent chose**:

In [ ]:
def text_of(msg):
    c = msg.content
    if isinstance(c, str):
        return c
    return "".join(p.get("text", "") if isinstance(p, dict) else str(p) for p in c)

def ask(question):
    print("QUESTION:", question, "\n")
    result = agent.invoke(
        {"messages": [{"role": "user", "content": question}]},
        config={"recursion_limit": 15},          # guardrail: cap the number of steps
    )
    for m in result["messages"][1:]:
        if m.type == "ai" and m.tool_calls:
            for call in m.tool_calls:
                print(f"  ACTION      -> {call['name']}({call['args']})")
        elif m.type == "tool":
            print(f"  OBSERVATION <- {text_of(m)[:250]}")
    print("\nANSWER:", text_of(result["messages"][-1]))

## Step 10 · Ask questions

**Graph question** — expect `find_entity` / `query_graph`:

In [ ]:
ask("Where does Alice work, and since when?")

**Document question** — expect `search_documents`:

In [ ]:
ask("What is the GraphRAG project for?")

**Multi-hop** — expect several `query_graph` steps:

In [ ]:
ask("Which company has an office in the city where Bob lives, and who from that company lives there?")

**Mixed** — expect both tools: graph for *who*, documents for *what*:

In [ ]:
ask("Who works on the project that uses Gemini, and what does the Gemini project do?")

## Try it yourself
- Lowercase names: `ask("who studied at iit hyderabad?")`
- A question the data can't answer: `ask("What is Alice's salary?")` — does it admit it?
- Ask it to delete data — `query_graph` should reject it

**GraphRAG chain vs Agentic GraphRAG**
| GraphRAG chain (Day 11) | Agentic GraphRAG |
|---|---|
| Always one Cypher query | Chooses vector search, Cypher, or both |
| Fails quietly on a bad query | Reads the error and retries |
| Graph facts only | Graph facts + original text |
| 2 LLM calls | Several LLM calls (slower, more quota) |

**If something fails**
- *429 / quota error* → wait a minute; the agent and indexing both use several calls
- *Embedding model not found* → set `GEMINI_EMBED_MODEL` in `.env` to an embedding model listed in AI Studio
- *GraphRecursionError* → the agent hit the 15-step cap; rephrase the question
- *APOC error* → install APOC in Neo4j Desktop and restart the database